### Connection MYSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [ ]:
import pandas as pd

### AKM

In [ ]:
akmQuery=text("""
WITH students AS (
    SELECT
        registers.id,
        registers.uuid,
        registers.nim,
        prodi_registers.name AS prodi_student,
        id_jenis_keluar AS jenis_keluar_student
    FROM
        mahasiswa_registers AS registers
    JOIN 
        prodi AS prodi_registers ON registers.prodi_uuid = prodi_registers.uuid
    WHERE
        registers.deleted_at IS NULL 
        AND registers.is_external = 0 
        AND registers.is_archive = 0
),
akm AS (
    SELECT
        s.nim,
        s.prodi_student,
        s.jenis_keluar_student,
        cuti.semester_start AS semester_cuti_start,
        keluar.semester_code AS semester_leave,
        keluar.jeniskeluar_code,
        kelas_laporan.semester_code AS semester_class_report,
		mbkm_activities.semester_code AS semester_mbkm
    FROM students s
    -- Join Cuti
    LEFT JOIN mahasiswa_cuti AS cuti 
        ON cuti.mahasiswa_register_uuid = s.uuid 
        AND cuti.deleted_at IS NULL 
        AND cuti.latest_status = 'approved'
    -- Join Keluar
    LEFT JOIN mahasiswa_keluar_details AS keluar_details 
        ON keluar_details.mahasiswa_register_uuid = s.uuid
    LEFT JOIN mahasiswa_keluar AS keluar 
        ON keluar_details.keluar_id = keluar.id
    -- Join Nilai & Kelas Laporan
    LEFT JOIN mahasiswa_nilais AS nilai 
        ON nilai.mahasiswa_register_id = s.id 
        AND nilai.deleted_at IS NULL
    LEFT JOIN kelas 
        ON nilai.kelas_id = kelas.id 
        AND kelas.deleted_at IS NULL
    LEFT JOIN kelas_laporan 
        ON kelas.kelas_laporan_uuid = kelas_laporan.uuid  
        AND kelas_laporan.deleted_at IS NULL
		-- Join MBKM
	LEFT JOIN mbkm_mahasiswa_applications AS mbkm_mahasiswa
		ON mbkm_mahasiswa.mahasiswa_register_uuid = s.uuid
	LEFT JOIN mahasiswa_activities AS mbkm_activities
		ON mbkm_mahasiswa.mahasiswa_activity_uuid = mbkm_activities.uuid
)
SELECT
	nim,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_cuti_start,
	semester_mbkm
FROM
    akm
GROUP BY
	nim,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_cuti_start,
	semester_mbkm
""")

In [ ]:
akmDf = pd.read_sql(akmQuery, siakadEngine)

In [ ]:
akmDf.info()

In [ ]:
akmDf

### Nilai Outlier

In [ ]:
outlier = (akmDf['jenis_keluar_student'] != akmDf['jeniskeluar_code']) & (
    akmDf['jenis_keluar_student'].notna()
)

In [ ]:
akmDf.loc[outlier, ['nim', 'prodi_student', 'jenis_keluar_student', 'jeniskeluar_code']].drop_duplicates()

In [ ]:
akmClearDf = akmDf[~outlier].copy()

In [ ]:
akmClearDf

### Master Semester

In [ ]:
semesterQuery = text("""
SELECT
    code, year_of_study
FROM
    semesters
WHERE
    year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
     year_of_study <= ( YEAR ( CURDATE())) AND
     semester != 3
""")

semesterDf = pd.read_sql(semesterQuery, siakadEngine)

In [ ]:
semesterDf

### Unique Valid Student

In [ ]:
studentDf = akmClearDf.loc[
    (akmDf['semester_leave'].isna()) |
    (
        (akmDf['semester_leave'] >= semesterDf['code'].iloc[0]) &
        (akmDf['semester_leave'] <= semesterDf['code'].iloc[-1])
    ),
    ['nim', 'prodi_student', 'semester_leave']
].drop_duplicates()

In [ ]:
studentDf

### Cross Join Semester x Student

In [ ]:
studentMergeDf = semesterDf.merge(studentDf, how="cross")

In [ ]:
studentMergeDf = studentMergeDf[
    (studentMergeDf["semester_leave"].isna()) |
    (studentMergeDf["code"] <= studentMergeDf["semester_leave"])
].drop(columns=['semester_leave'])

studentMergeDf['status'] = 'nonaktif'

In [ ]:
studentMergeDf

### Student Have Semester Laporan

In [ ]:
studentSemesterLaporan = akmClearDf.loc[
    (akmClearDf['semester_class_report'].notna()) &
    (akmClearDf['semester_class_report'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_class_report']
].rename(columns={
    'semester_class_report': 'code'
}).drop_duplicates().copy()

In [ ]:
studentSemesterLaporan['status_laporan'] = 'Aktif'

In [ ]:
studentSemesterLaporan

### Student Have MBKM 

In [ ]:
studentMbkm = akmClearDf.loc[
    (akmClearDf['semester_mbkm'].notna()) &
    (akmClearDf['semester_mbkm'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_mbkm']
].rename(columns={
    'semester_mbkm': 'code'
}).drop_duplicates().copy()

In [ ]:
studentMbkm['status_mbkm'] = 'aktif'

In [ ]:
studentMbkm

### Student Have Cuti

In [ ]:
studentCuti = akmClearDf.loc[
    (akmClearDf['semester_cuti_start'].notna()) &
    (akmClearDf['semester_cuti_start'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_cuti_start']
].rename(columns={
    'semester_cuti_start': 'code'
}).drop_duplicates().copy()

In [ ]:
studentCuti['status_cuti'] = 'cuti'

In [ ]:
studentCuti

### Student Graduate

In [ ]:
studentGraduate = akmClearDf.loc[
    (akmClearDf['semester_leave'].notna()) &
    (akmClearDf['semester_leave'] >= semesterDf['code'].iloc[0]) &
    (akmClearDf['jeniskeluar_code'] == '1'),
    ['nim', 'semester_leave']
].rename(columns={
    'semester_leave': 'code'
}).drop_duplicates().copy()

In [ ]:
studentGraduate['status_graduate'] = 'lulus';

In [ ]:
studentGraduate

### Final AKM

In [ ]:
finalAkmDf = studentMergeDf.merge(
    studentSemesterLaporan, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentMbkm, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentCuti, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentGraduate, on=['nim', 'code'], how='left'
)

In [ ]:
finalAkmDf['status'] = (
    finalAkmDf.get('status_laporan')
    .combine_first(finalAkmDf.get('status_mbkm'))
    .combine_first(finalAkmDf.get('status_cuti'))
    .combine_first(finalAkmDf.get('status_graduate'))
    .combine_first(finalAkmDf.get('status'))   
)

cols_to_drop = [c for c in finalAkmDf.columns if c.startswith('status_') and c != 'status']
finalAkmDf = finalAkmDf.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
finalAkmDf[
    (finalAkmDf['prodi_student'] == 'Matematika') &
    (finalAkmDf['code'] == '20221') &
    (finalAkmDf['status'] == 'nonaktif')
]